# Spotify Song Clustering and Playlist Discovery

Group tracks by audio characteristics and turn clusters into coherent playlist seeds.

**Portfolio category:** Clustering

**Data mode:** Verified demo mode

This notebook keeps labels out of fitting wherever labels exist, uses deterministic seeds,
reports unsupervised-specific diagnostics, and avoids hard-coded results.

## 1. Project setup

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

## 2. Spotify-style audio features

In [ ]:
features = ["danceability", "energy", "acousticness", "instrumentalness", "valence", "tempo"]
genre_profiles = {
    "acoustic": [0.42, 0.30, 0.82, 0.25, 0.48, 92],
    "dance": [0.83, 0.82, 0.10, 0.05, 0.72, 126],
    "hip_hop": [0.76, 0.70, 0.16, 0.03, 0.58, 101],
    "rock": [0.52, 0.88, 0.08, 0.08, 0.55, 134],
    "ambient": [0.28, 0.25, 0.68, 0.86, 0.34, 78],
}
rows = []
for genre, profile in genre_profiles.items():
    values = rng.normal(profile, [0.08, 0.08, 0.09, 0.08, 0.10, 8], size=(150, 6))
    frame = pd.DataFrame(values, columns=features)
    frame[features[:-1]] = frame[features[:-1]].clip(0, 1)
    frame["tempo"] = frame["tempo"].clip(50, 210)
    frame["hidden_genre"] = genre
    rows.append(frame)
tracks = pd.concat(rows, ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
tracks["track_name"] = [f"Track {i + 1:04d}" for i in range(len(tracks))]
display(tracks.head())

## 3. Feature quality and distribution

In [ ]:
display(tracks[features].describe().T)
tracks[features].hist(figsize=(12, 7), bins=25)
plt.tight_layout()

## 4. Scale features and select k

In [ ]:
X = StandardScaler().fit_transform(tracks[features])
rows = []
for k in range(2, 9):
    labels = KMeans(n_clusters=k, n_init=25, random_state=RANDOM_STATE).fit_predict(X)
    rows.append({
        "k": k,
        "silhouette": silhouette_score(X, labels),
        "davies_bouldin": davies_bouldin_score(X, labels),
    })
scores = pd.DataFrame(rows)
display(scores.round(3))
best_k = int(scores.sort_values(["silhouette", "davies_bouldin"], ascending=[False, True]).iloc[0]["k"])
model = KMeans(n_clusters=best_k, n_init=40, random_state=RANDOM_STATE)
tracks["cluster"] = model.fit_predict(X)

## 5. Evaluate cluster structure

In [ ]:
genre_codes = tracks["hidden_genre"].astype("category").cat.codes
display(pd.Series({
    "selected_k": best_k,
    "silhouette": silhouette_score(X, tracks["cluster"]),
    "davies_bouldin": davies_bouldin_score(X, tracks["cluster"]),
    "adjusted_rand_vs_hidden_genre": adjusted_rand_score(genre_codes, tracks["cluster"]),
}).to_frame("value"))

## 6. Cluster profiles

In [ ]:
profile = tracks.groupby("cluster")[features].mean()
display(profile.round(3))
sns.heatmap(profile[features[:-1]], cmap="vlag", center=0.5)
plt.title("Audio feature profiles")
plt.tight_layout()

## 7. Playlist discovery view

In [ ]:
projection = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X)
sns.scatterplot(x=projection[:, 0], y=projection[:, 1], hue=tracks["cluster"], palette="tab10", alpha=0.65)
plt.title("Track clusters in PCA space")
plt.tight_layout()

## 8. Sample playlist seeds

In [ ]:
playlist_seeds = tracks.groupby("cluster", group_keys=False).apply(
    lambda frame: frame.sample(min(5, len(frame)), random_state=RANDOM_STATE)
)[["cluster", "track_name", "hidden_genre", *features]]
display(playlist_seeds)

## 9. Key findings

Profile clusters by audio features before assigning playlist names; hidden genres are used only to audit the demonstration.

## 10. Interpretation and responsible use

Treat the output as exploratory evidence, not ground truth. For spotify song clustering and playlist discovery,
validate stability on newer data, inspect edge cases, and review domain risks before
turning clusters, rankings or anomaly scores into decisions.

## 11. Next steps

- Replace demonstration data with a versioned, licensed dataset.
- Track data quality, drift and stability across repeated runs.
- Add domain-specific review before deployment.
- Package inference only after reproducibility and privacy checks pass.

All numeric results are generated at execution time; none are hard-coded.